# PTCG AI Battle — Static-Deck Great Tusk / Crustle

## A transparent, evaluator-safe Library-Out policy

This notebook packages a public Great Tusk / Crustle Library-Out policy for
Kaggle-native Pokémon TCG AI Battle evaluation. It is derived from
[SOUTA Sakurai's public 1208 policy](https://www.kaggle.com/code/soutasakurai/max-elo-1208-libraryout-w-crustle-great-tusk).

The strategic core is unchanged:

1. pressure the opponent's deck with Great Tusk and Explorer's Guidance;
2. use Crustle plus Neutralization Zone as a non-ex defensive line;
3. use search, recovery, and disruption to maintain the chosen plan.

### Why this packaging differs

The competition loads raw Python and invokes the last callable created by the
source. It can also run from a directory where a relative `deck.csv` is not
available. This version makes the *same public 60-card deck* a literal in the
source and verifies both evaluator conditions before creating the archive.

The cited policy's historical rating is provenance, not a prediction. Only a
completed competition evaluation of the exact archive below is score evidence.


In [ ]:
from pathlib import Path
import base64
import hashlib
import os
import shutil
import sys
import tarfile

WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
PAYLOADS = {"deck.csv": "NTgKNTgKNTgKNTgKMzQ0CjM0NAozNDQKMzQ0CjM0NQozNDUKMTE0MgoxMTQyCjExNDIKMTE0MgoxMTUyCjExNTIKMTE1MgoxMTUyCjEwODYKMTA4NgoxMDg2CjEwODYKMTEyMgoxMTIyCjExMjIKMTEyMgoxMTIxCjExMjMKMTEyMwoxMTIzCjExMjMKMTE5NwoxMTk3CjExOTcKMTE5NwoxMTg1CjExODUKMTE4NQoxMTg1CjExODIKMTE4MgoxMTgyCjExODIKMTIwNAoxMjA0CjExOTQKMTE5NAoxMjQ3CjExNDcKMjAKMjAKMjAKMjAKMTEKMTEKMTEKMTEKMzQ1CjM0NQo2MDcK", "main.py": "..."}
SOURCE_SHA256 = '8f2fa9c432642cd07b1fa10246aa200bbd2713b94cbc8a98d902419ff4ad18c8'
DECK_SHA256 = '6415396d35c0f4b3d69ee6c231337968cc9f2d5d0767de801346d6f412c18e62'
EXPECTED_DECK = [58, 58, 58, 58, 344, 344, 344, 344, 345, 345, 1142, 1142, 1142, 1142, 1152, 1152, 1152, 1152, 1086, 1086, 1086, 1086, 1122, 1122, 1122, 1122, 1121, 1123, 1123, 1123, 1123, 1197, 1197, 1197, 1197, 1185, 1185, 1185, 1185, 1182, 1182, 1182, 1182, 1204, 1204, 1194, 1194, 1247, 1147, 20, 20, 20, 20, 11, 11, 11, 11, 345, 345, 607]

for name, payload in PAYLOADS.items():
    target = WORK / name
    target.write_bytes(base64.b64decode(payload.encode('ascii')))
    print(f'{name}: {target.stat().st_size:,} bytes sha256={hashlib.sha256(target.read_bytes()).hexdigest()}')

assert hashlib.sha256((WORK / 'main.py').read_bytes()).hexdigest() == SOURCE_SHA256
assert hashlib.sha256((WORK / 'deck.csv').read_bytes()).hexdigest() == DECK_SHA256
assert len(EXPECTED_DECK) == 60
assert [int(v) for v in (WORK / 'deck.csv').read_text().splitlines() if v.strip()] == EXPECTED_DECK
print('static deck cards:', len(EXPECTED_DECK), '| unique IDs:', len(set(EXPECTED_DECK)))


## Package the exact source, deck, and official engine

In [ ]:
def find_cg_source() -> Path:
    candidates = (
        '/kaggle/input/competitions/pokemon-tcg-ai-battle/sample_submission/cg',
        '/kaggle/input/competitions/pokemon-tcg-ai-battle/sample_submission/sample_submission/cg',
        '/kaggle/input/pokemon-tcg-ai-battle/sample_submission/cg',
        '/kaggle/input/pokemon-tcg-ai-battle/sample_submission/sample_submission/cg',
        # Optional local-execution bridge. Kaggle has no such environment
        # variable, so public runs still prove the official-input path.
        os.environ.get('PTCG_CG_SOURCE', ''),
    )
    for raw in candidates:
        candidate = Path(raw)
        if (candidate / 'api.py').is_file() and (candidate / 'libcg.so').is_file():
            return candidate
    raise FileNotFoundError('Attach the official pokemon-tcg-ai-battle competition input.')

build = WORK / 'tusk1208_static_submission'
if build.exists():
    shutil.rmtree(build)
build.mkdir()
for name in ('main.py', 'deck.csv'):
    shutil.copy2(WORK / name, build / name)
shutil.copytree(find_cg_source(), build / 'cg')
for cache in build.rglob('__pycache__'):
    shutil.rmtree(cache)
for bytecode in list(build.rglob('*.pyc')) + list(build.rglob('*.pyo')):
    bytecode.unlink()
print('build:', build)


## Evaluator-faithful raw-source contract check

In [ ]:
# This mirrors Kaggle's raw-agent loader: exec with an empty namespace,
# then use the final callable created by that source.  __file__ is deliberately absent.
old_cwd = Path.cwd()
sys.path.insert(0, str(build))
os.chdir(build)
try:
    raw_namespace = {}
    raw_source = (build / 'main.py').read_text(encoding='utf-8')
    exec(compile(raw_source, '/kaggle_simulations/agent/main.py', 'exec'), raw_namespace)
    selected = [value for value in raw_namespace.values() if callable(value)][-1]
    assert selected is raw_namespace['agent'], selected

    class Struct(dict):
        def __init__(self, **entries):
            super().__init__(entries)
            self.__dict__.update(entries)

    startup = Struct(current=None, select=None, logs=[], search_begin_input=None)
    chosen_deck = selected(startup)
    assert chosen_deck == EXPECTED_DECK
    assert len(chosen_deck) == 60
    print('selected callable:', selected.__name__)
    print('raw-exec startup deck:', len(chosen_deck), 'cards')
finally:
    os.chdir(old_cwd)
    sys.path.remove(str(build))


## Official-engine self-game smoke test

In [ ]:
old_cwd = Path.cwd()
sys.path.insert(0, str(build))
os.chdir(build)
try:
    raw_namespace = {}
    exec(compile((build / 'main.py').read_text(encoding='utf-8'), '/kaggle_simulations/agent/main.py', 'exec'), raw_namespace)
    agent = [value for value in raw_namespace.values() if callable(value)][-1]
    from cg.api import to_observation_class
    from cg.game import battle_finish, battle_select, battle_start

    observation, started = battle_start(EXPECTED_DECK, EXPECTED_DECK)
    if observation is None:
        raise RuntimeError(f'battle_start failed: {{started}}')
    steps = 0
    try:
        while steps < 600:
            current = to_observation_class(observation).current
            if current.result >= 0:
                break
            choices = agent(observation)
            select = observation['select']
            assert isinstance(choices, list)
            assert int(select['minCount']) <= len(choices) <= int(select['maxCount'])
            assert all(isinstance(i, int) and 0 <= i < len(select['option']) for i in choices)
            observation = battle_select(choices)
            steps += 1
        else:
            raise TimeoutError('self-game exceeded 600 legal selections')
    finally:
        battle_finish()
    print('official-engine self-game completed in', steps, 'selections')
finally:
    os.chdir(old_cwd)
    sys.path.remove(str(build))


## Kaggle-native handoff

In [ ]:
# The raw-agent smoke test imports the copied engine and can recreate
# bytecode caches, so clean once more immediately before archiving.
for cache in build.rglob('__pycache__'):
    shutil.rmtree(cache)
for bytecode in list(build.rglob('*.pyc')) + list(build.rglob('*.pyo')):
    bytecode.unlink()

archive = WORK / 'submission.tar.gz'
if archive.exists():
    archive.unlink()
with tarfile.open(archive, 'w:gz') as tar:
    tar.add(build / 'main.py', arcname='main.py')
    tar.add(build / 'deck.csv', arcname='deck.csv')
    tar.add(build / 'cg', arcname='cg')
with tarfile.open(archive, 'r:gz') as tar:
    members = set(tar.getnames())
required = {'main.py', 'deck.csv', 'cg/api.py', 'cg/game.py', 'cg/sim.py', 'cg/libcg.so'}
assert required <= members, sorted(required - members)
assert not any('__pycache__' in name or name.endswith(('.pyc', '.pyo')) for name in members)
print('archive:', archive, '| bytes:', archive.stat().st_size)
print('Submit this exact Kaggle-produced archive through the competition UI.')


### Reproducibility notes

- Source is attributed and byte-checked.
- The only implementation change is replacing a runtime deck-file read with
  the same 60 public card IDs as a literal.
- The official `cg` engine is copied from the attached competition input.
- The notebook verifies raw callable selection, pre-game deck output, archive
  contents, and a legal self-game before handoff.
- No external services, hidden state, private data, or network access are used.
